# Part B — Install and Load Local Qwen3.5-4B

**Goal:** Prepare a local, multimodal model for the bookstore tool-calling exercises in Parts C–H. This part **does not use tools**. The bookstore database and API built in Part A remain unchanged.

**Learning outcomes**

- Differentiate the **model**, **tokenizer**, and **multimodal processor**.
- Explain why we use a GPU and **4-bit NF4 quantization**.
- Explain the difference between `!python filename.py` (new process) and `%run -i filename.py` (same notebook kernel).
- Confirm the model can generate a basic answer before connecting any bookstore functions.

Check that the GPU runtime selected during preparation is still active before continuing.

All code is saved to a Python file first with `%%writefile`.


## B1 — Check the environment

**Why?** A 4-billion-parameter model requires much more memory and computation than our SQLite examples. Colab's GPU makes 4-bit local inference practical. Running this check first avoids spending time downloading weights into a CPU-only session.

**Pattern:** write a file, execute the file, inspect the output.

In [ ]:
%%writefile qwen_gpu_check.py
"""Part B1 — Confirm this runtime has a suitable GPU before downloading Qwen."""
import platform
import torch

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
if not torch.cuda.is_available():
    raise SystemExit(
        "CUDA GPU not detected. In Colab choose Runtime > Change runtime type "
        "> T4 GPU (or another NVIDIA GPU), reconnect, and run this cell again."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"Free GPU memory: {free_bytes / 2**30:.2f} GiB")
print(f"Total GPU memory: {total_bytes / 2**30:.2f} GiB")
print("Preflight: PASS")


In [ ]:
!python qwen_gpu_check.py

**B1 questions**

1. **What** GPU was assigned? How much total GPU memory is available?
2. **Why** is `torch.cuda.is_available()` checked *before* downloading the model?
3. **What** is the difference between GPU **total**, **free**, and **allocated** memory?

**Try:** Run the GPU check again after B3. What changed? Why?

## B2 — Install the model dependencies

Install transformers, accelerate, bitsandbytes, and pillow for later real-image input. We do **not** install LangChain here; Part C introduces tools first.

The Python install script runs as an ordinary subprocess because it only changes packages on disk. It doesn't need to share notebook variables.

> **Colab note:** If you have already imported `transformers` in this notebook before upgrading it, restart the runtime after installation, then rerun Part A (if needed) and Part B.

In [ ]:
%%writefile qwen_install.py
"""Part B2 — Install the packages necessary for Qwen3.5-4B in Colab.

Run as a separate process: python qwen_install.py
Note: PyTorch is already supplied by the Colab GPU environment, so we do
not force a new torch installation.
"""
import subprocess
import sys

packages = [
    "transformers>=4.57.0",  # install/upgrade to the newest available version
    "accelerate",
    "bitsandbytes",        # for NF4 4-bit quantization on NVIDIA GPUs
    "pillow",              # will be needed for real book-cover images later
]
command = [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *packages]
print("Installing: " + ", ".join(packages), flush=True)
subprocess.check_call(command)
print("Dependencies installed. If the notebook already imported transformers, restart its runtime once.")


In [ ]:
!python qwen_install.py

**B2 questions**

1. **What** does `transformers` provide? What does `accelerate` provide?
2. **Why** is `bitsandbytes` useful when GPU memory is limited?
3. **How** does installing dependencies differ from loading model weights?
4. **Why** is `!python qwen_install.py` appropriate here, while `%run -i` is preferable for the model loader?

## B3 — Load Qwen3.5-4B (4-bit, multimodal)

We use [`Qwen/Qwen3.5-4B`](https://huggingface.co/Qwen/Qwen3.5-4B). We load it through Hugging Face's **multimodal** model class and **AutoProcessor**, instead of treating it as text-only. This will let us send a real book-cover image to Qwen in a later part.

- `MODEL_ID`: identifies the pretrained model in Hugging Face.
- `processor`: prepares text now, and text + images later.
- `tokenizer = processor.tokenizer`: maps text to/from token IDs.
- `model`: performs inference on GPU.
- **NF4 4-bit quantization** reduces memory needs, with possible accuracy/performance trade-offs.

**Critical:** Use `%run -i qwen_load.py` to retain `model`, `processor`, and `tokenizer` in the notebook kernel. `!python qwen_load.py` would load the model in a separate process and then discard it when that process exits.

**Don't rerun loading unnecessarily.** This script reuses an existing in-kernel model, but a Colab restart clears the in-memory model and may require weights to be reloaded.

In [ ]:
%%writefile qwen_load.py
"""Part B3 — Load the *multimodal* Qwen3.5-4B model in 4-bit mode.

In Jupyter/Colab run: %run -i qwen_load.py
Why -i? It shares the current notebook namespace, keeping `model`,
`processor`, and `tokenizer` accessible to Parts C–H.

Do not call this with !python: the model would be discarded when the
subprocess ends. If `model` already exists, we reuse it to avoid reloading.
"""
import os
from pathlib import Path
import torch
from transformers import AutoProcessor, BitsAndBytesConfig

try:
    from transformers import AutoModelForMultimodalLM
except ImportError as exc:
    raise RuntimeError(
        "AutoModelForMultimodalLM is unavailable. Update Transformers with "
        "!python qwen_install.py, restart the runtime if needed, then retry."
    ) from exc

MODEL_ID = "Qwen/Qwen3.5-4B"
CACHE_DIR = os.environ.get(
    "HF_HOME", "/content/hf_cache" if Path("/content").exists() else "./hf_cache"
)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this 4-bit teaching setup.")

if "model" in globals() and "processor" in globals():
    print("Qwen model already loaded in this notebook kernel; reusing it.")
else:
    print(f"Loading {MODEL_ID} with 4-bit NF4 weights (cache: {CACHE_DIR})...", flush=True)
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    # AutoProcessor carries the tokenizer AND the image pre-processing path.
    # This is essential for the genuine multimodal task in a later Part.
    processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
        dtype=torch.float16,
        cache_dir=CACHE_DIR,
        low_cpu_mem_usage=True,
    )
    model.eval()

tokenizer = processor.tokenizer
print("Model ID:", MODEL_ID)
print("Model class:", type(model).__name__)
print("Processor class:", type(processor).__name__)
print("Model type:", model.config.model_type)
print("Quantized in 4-bit:", bool(getattr(model, "is_loaded_in_4bit", False)))
print(f"GPU allocated: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")
print("Model is ready in this notebook kernel (text and image inputs supported).")


In [ ]:
%run -i qwen_load.py

**B3 questions**

1. **What** is the difference between a model's **weights** and its **tokenizer**?
2. **Why** do we use `AutoProcessor` rather than relying exclusively on `AutoTokenizer`?
3. **How** might 4-bit quantization affect GPU memory, model precision, and generation quality?
4. **What** would happen if the load were called with `!python` and the next notebook cell tried to access `model`?
5. **Why** don't we need a Groq API key to run this local model?

## B4 — Verify text generation (no tools yet)

Before connecting tools to the model, test a minimal, self-contained generation. In this part the model has **no connection to SQLite**. A fluent answer does not mean it can read the bookstore database.

We set `enable_thinking=False` for a short direct classroom answer, and limit `max_new_tokens` to avoid unexpectedly lengthy output. The same multimodal model and processor stay in memory for later parts.

In [ ]:
%%writefile qwen_smoke_test.py
"""Part B4 — A simple text-only check; no bookstore tools yet.

In Jupyter/Colab run: %run -i qwen_smoke_test.py
Assumes %run -i qwen_load.py was executed in this kernel.
"""
import torch

if "model" not in globals() or "processor" not in globals():
    raise RuntimeError("First execute %run -i qwen_load.py in the notebook.")

QUESTION = "In one short sentence, what is a book genre?"
messages = [
    {"role": "user", "content": [{"type": "text", "text": QUESTION}]}
]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    enable_thinking=False,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

print("Prompt:", QUESTION)
print("Prompt tokens:", inputs["input_ids"].shape[-1])
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,  # deterministic classroom smoke test (not a benchmark)
        pad_token_id=tokenizer.eos_token_id,
    )
new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
answer = processor.decode(new_tokens, skip_special_tokens=True)
# Qwen normally emits no thinking text with enable_thinking=False;
# if a runtime still returns <think> tags, keep just the answer.
if "</think>" in answer:
    answer = answer.split("</think>", 1)[1]
answer = answer.strip()
print("Model output:", answer)
print("New tokens:", len(new_tokens))
print("This part tested generation only—not tools, SQL, or fact retrieval.")


In [ ]:
%run -i qwen_smoke_test.py

**B4 questions**

1. **What** is the difference between a *prompt token* and a *generated token*?
2. **Why** are only the newly generated tokens decoded and printed?
3. **How** does `max_new_tokens` affect the model's response?
4. **Can** the model reliably tell you the exact number of books in `bookstore.db` at this point? Why or why not?
5. **What** changes when `enable_thinking=False`? Why might we prefer that during a quick smoke test?

**Exercise:** Change `QUESTION` in `qwen_smoke_test.py` to ask for a two-sentence explanation of a bestseller. Rerun the `%%writefile` cell and `%run -i qwen_smoke_test.py`. **Do not rerun B3 to change only the prompt.** Compare the result with the original and record any deviations from the requested format.

## B — Exit ticket

Complete the sentence and answer the check before proceeding to Part C:

> In Part A, `search_books` was a regular ______ function that queried SQLite. In Part B, we loaded a ______ model, but it cannot call `search_books` because we have not yet ______ it as a tool.

**Ready for Part C when:** (1) the GPU check passes, (2) the model and processor load, (3) a simple answer is generated, and (4) you can explain `%run -i` versus `!python`.

**Troubleshooting:** If CUDA isn't available, change Colab runtime type to GPU. If the model class cannot be imported, update Transformers and restart the runtime. If model loading runs out of GPU memory, restart with a clean GPU runtime and run B3 only once. If a previous part was executed on another Colab runtime, its Python variables won't be available here.

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.free,memory.total --format=csv